In [1]:
# Performance config
import os

CPU_THREADS = 32  # user has 32 threads
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

print(f"CPU threads set to: {CPU_THREADS}")

# GPU availability
try:
    import torch
except Exception:
    torch = None
    print("Torch not available; GPU checks disabled.")
else:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available; will use CPU backend.")


CPU threads set to: 32
CUDA device: NVIDIA GeForce RTX 3090


# Kvasir-VQA x1 — Text-only baselines

Train text-only classifiers on questions:
- Yes/No binary classification
- Open-ended top-K answer classification

Outputs saved to `2_modeling/01_text_only/out/`.


In [2]:
from pathlib import Path
import json
import random
import re

import numpy as np
import pandas as pd

# Try GPU stack first (cuML); fall back to sklearn on CPU
HAS_CUML = False
try:
    from cuml.feature_extraction.text import TfidfVectorizer as cuTfidfVectorizer
    from cuml.linear_model import LogisticRegression as cuLogisticRegression
    HAS_CUML = True
except Exception:
    cuTfidfVectorizer = None
    cuLogisticRegression = None

HAS_CUDF = False
try:
    import cudf
    HAS_CUDF = True
except Exception:
    cudf = None

from sklearn.feature_extraction.text import TfidfVectorizer as skTfidfVectorizer
from sklearn.linear_model import LogisticRegression as skLogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    top_k_accuracy_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)


In [3]:
# Paths & config

def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "01_text_only" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
TOP_K = 200
MAX_FEATURES = 5000
NGRAM_RANGE = (1, 2)

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)


Data root: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/01_text_only/out


In [4]:
# Load metadata
meta = pd.read_csv(META_CSV)

# Normalize text fields
meta["question_norm"] = meta["question"].fillna("").astype(str).str.lower().str.strip()
meta["answer_norm"] = meta["answer"].fillna("").astype(str).str.lower().str.strip()

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 143594, 'val': 0, 'test': 15955}


In [5]:
# Summary of dataset coverage for this notebook
print("Total rows:", len(meta))
if "split" in meta:
    print("Split counts:", meta["split"].value_counts().to_dict())
if "img_id" in meta:
    print("Unique images:", meta["img_id"].nunique())

# Yes/No coverage
ans_norm = (
    meta["answer"].astype(str).str.lower().str.strip()
    .str.replace(r"[^a-z]", "", regex=True)
)
yn_mask = ans_norm.isin(["yes", "no"])
print("Yes/No rows:", int(yn_mask.sum()))
if "img_id" in meta:
    print("Unique images (Yes/No):", meta.loc[yn_mask, "img_id"].nunique())

# Top-K coverage (from train split only)
if "answer_norm" in train_df:
    topk_answers = train_df["answer_norm"].value_counts().head(TOP_K).index
    train_k = train_df[train_df["answer_norm"].isin(topk_answers)]
    val_k = val_df[val_df["answer_norm"].isin(topk_answers)] if len(val_df) else val_df
    test_k = test_df[test_df["answer_norm"].isin(topk_answers)] if len(test_df) else test_df
    print("Top-K answers:", len(topk_answers))
    print({"train": len(train_k), "val": len(val_k), "test": len(test_k)})
    if "img_id" in meta:
        print("Unique images (Top-K train):", train_k["img_id"].nunique())
        print("Unique images (Top-K test):", test_k["img_id"].nunique())


Total rows: 159549
Split counts: {'train': 143594, 'test': 15955}
Unique images: 6449
Yes/No rows: 0
Unique images (Yes/No): 0
Top-K answers: 200
{'train': 38424, 'val': 0, 'test': 4252}
Unique images (Top-K train): 6106
Unique images (Top-K test): 2694


In [6]:
# Utilities

def _to_numpy(x):
    try:
        import cupy as cp
        if isinstance(x, cp.ndarray):
            return cp.asnumpy(x)
    except Exception:
        pass
    return np.asarray(x)

def _as_cudf_series(values):
    if not HAS_CUDF:
        return values
    try:
        return cudf.Series(values)
    except Exception:
        return cudf.Series(list(values))

def _safe_predict_proba(clf, X):
    try:
        proba = clf.predict_proba(X)
    except Exception:
        return None
    return _to_numpy(proba)

def _compute_metrics(y_true, y_pred, y_proba, labels):
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

    pr, rc, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(len(labels))), zero_division=0
    )
    per_class = {}
    for i, name in enumerate(labels):
        per_class[str(name)] = {
            "precision": float(pr[i]),
            "recall": float(rc[i]),
            "f1": float(f1[i]),
            "support": int(support[i]),
        }
    metrics["per_class"] = per_class

    if y_proba is not None:
        # Top-K accuracy (if enough classes)
        n_classes = y_proba.shape[1]
        for k in (3, 5):
            if n_classes >= k:
                metrics[f"top_{k}_accuracy"] = float(
                    top_k_accuracy_score(
                        y_true, y_proba, k=k, labels=list(range(n_classes))
                    )
                )
        # Binary-only probability metrics
        if n_classes == 2:
            pos = y_proba[:, 1]
            try:
                metrics["roc_auc"] = float(roc_auc_score(y_true, pos))
                metrics["pr_auc"] = float(average_precision_score(y_true, pos))
                metrics["brier"] = float(brier_score_loss(y_true, pos))
            except Exception:
                pass
    return metrics

CUDA_AVAILABLE = (torch is not None and torch.cuda.is_available())
if CUDA_AVAILABLE and not (HAS_CUML and HAS_CUDF):
    missing = []
    if not HAS_CUML:
        missing.append("cuml")
    if not HAS_CUDF:
        missing.append("cudf")
    raise RuntimeError(
        f"CUDA is available but required GPU deps are missing: {', '.join(missing)}. Install these to run on GPU."
    )

USE_GPU = (CUDA_AVAILABLE and HAS_CUML and HAS_CUDF)
print("Backend:", "GPU (cuML)" if USE_GPU else "CPU (scikit-learn)")

def fit_eval_text_classifier(train_df, val_df, test_df, label_col, out_prefix):
    if USE_GPU:
        vectorizer = cuTfidfVectorizer(max_features=MAX_FEATURES, ngram_range=NGRAM_RANGE)
        clf = cuLogisticRegression(max_iter=1000)

        labels = sorted(train_df[label_col].dropna().unique())
        label_to_id = {lab: i for i, lab in enumerate(labels)}
        id_to_label = {i: lab for lab, i in label_to_id.items()}

        def encode(df):
            y = df[label_col].map(label_to_id)
            mask = y.notna()
            df_enc = df[mask].reset_index(drop=True)
            return df_enc, y[mask].astype(int).values

        train_df_enc, y_train = encode(train_df)
        X_train = vectorizer.fit_transform(_as_cudf_series(train_df_enc["question_norm"]))
        clf.fit(X_train, y_train)

        def eval_split(df, split_name):
            df_enc, y_true = encode(df)
            if len(df_enc) == 0:
                return
            X = vectorizer.transform(_as_cudf_series(df_enc["question_norm"]))
            y_pred = _to_numpy(clf.predict(X)).astype(int)
            y_true = _to_numpy(y_true).astype(int)
            y_proba = _safe_predict_proba(clf, X)

            metrics = _compute_metrics(y_true, y_pred, y_proba, labels)
            report = classification_report(
                y_true, y_pred,
                labels=list(range(len(labels))),
                target_names=labels,
                output_dict=True,
                zero_division=0
            )

            pred_df = df_enc[["question", "answer", label_col]].copy()
            pred_df["pred"] = [id_to_label[i] for i in y_pred]
            pred_df.to_csv(OUT_DIR / f"{out_prefix}_pred_{split_name}.csv", index=False)

            with open(OUT_DIR / f"{out_prefix}_metrics_{split_name}.json", "w") as f:
                json.dump({"metrics": metrics, "report": report}, f, indent=2)

            cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
            cm_df = pd.DataFrame(cm, index=labels, columns=labels)
            cm_df.to_csv(OUT_DIR / f"{out_prefix}_confusion_{split_name}.csv")

            print(split_name, {k: v for k, v in metrics.items() if k != "per_class"})

        eval_split(train_df, "train")
        if len(val_df):
            eval_split(val_df, "val")
        if len(test_df):
            eval_split(test_df, "test")

        return clf, vectorizer

    # CPU fallback (only when CUDA is unavailable)
    vectorizer = skTfidfVectorizer(max_features=MAX_FEATURES, ngram_range=NGRAM_RANGE)
    clf = skLogisticRegression(max_iter=1000, n_jobs=CPU_THREADS)

    X_train = vectorizer.fit_transform(train_df["question_norm"])
    y_train = train_df[label_col].values
    clf.fit(X_train, y_train)

    labels = list(getattr(clf, "classes_", sorted(train_df[label_col].dropna().unique())))
    label_to_id = {lab: i for i, lab in enumerate(labels)}
    id_to_label = {i: lab for lab, i in label_to_id.items()}

    def encode(df):
        y = df[label_col].map(label_to_id)
        mask = y.notna()
        df_enc = df[mask].reset_index(drop=True)
        return df_enc, y[mask].astype(int).values

    def eval_split(df, split_name):
        df_enc, y_true = encode(df)
        if len(df_enc) == 0:
            return
        X = vectorizer.transform(df_enc["question_norm"])
        y_pred = [label_to_id[v] for v in clf.predict(X)]
        y_pred = np.asarray(y_pred).astype(int)
        y_proba = _safe_predict_proba(clf, X)

        metrics = _compute_metrics(y_true, y_pred, y_proba, labels)
        report = classification_report(
            y_true, y_pred,
            labels=list(range(len(labels))),
            target_names=labels,
            output_dict=True,
            zero_division=0
        )

        pred_df = df_enc[["question", "answer", label_col]].copy()
        pred_df["pred"] = [id_to_label[i] for i in y_pred]
        pred_df.to_csv(OUT_DIR / f"{out_prefix}_pred_{split_name}.csv", index=False)

        with open(OUT_DIR / f"{out_prefix}_metrics_{split_name}.json", "w") as f:
            json.dump({"metrics": metrics, "report": report}, f, indent=2)

        cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
        cm_df = pd.DataFrame(cm, index=labels, columns=labels)
        cm_df.to_csv(OUT_DIR / f"{out_prefix}_confusion_{split_name}.csv")

        print(split_name, {k: v for k, v in metrics.items() if k != "per_class"})

    eval_split(train_df, "train")
    if len(val_df):
        eval_split(val_df, "val")
    if len(test_df):
        eval_split(test_df, "test")

    return clf, vectorizer


Backend: GPU (cuML)


In [7]:
# Yes/No classification (question only)

def normalize_yesno(text):
    t = re.sub(r"[^a-z]", "", str(text).lower())
    if t in ("yes", "no"):
        return t
    return None

meta["answer_yesno"] = meta["answer"].apply(normalize_yesno)
yn_df = meta[meta["answer_yesno"].notna()].reset_index(drop=True)

train_yn = yn_df[yn_df["split"] == "train"].reset_index(drop=True)
val_yn = yn_df[yn_df["split"] == "validation"].reset_index(drop=True)
test_yn = yn_df[yn_df["split"] == "test"].reset_index(drop=True)

print("Yes/No rows:", len(yn_df))
if len(train_yn):
    fit_eval_text_classifier(train_yn, val_yn, test_yn, "answer_yesno", "yesno_text")


Yes/No rows: 0


In [8]:
# Top-K answer classification (question only)

# Build top-K answers from train split only to avoid leakage
answer_counts = train_df["answer_norm"].value_counts()
TOP_K_ANSWERS = answer_counts.head(TOP_K).index.tolist()

train_k = train_df[train_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
val_k = val_df[val_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
test_k = test_df[test_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)

print("Top-K answers:", len(TOP_K_ANSWERS))
print({"train": len(train_k), "val": len(val_k), "test": len(test_k)})

if len(train_k):
    fit_eval_text_classifier(train_k, val_k, test_k, "answer_norm", "topk_text")


Top-K answers: 200
{'train': 38424, 'val': 0, 'test': 4252}
train {'accuracy': 0.42988757026858215, 'macro_f1': 0.22174292001213558, 'weighted_f1': 0.33750658876964995, 'balanced_accuracy': 0.2516359004130564, 'top_3_accuracy': 0.75, 'top_5_accuracy': 0.8729439933374974}
test {'accuracy': 0.42238946378174974, 'macro_f1': 0.20410272410618321, 'weighted_f1': 0.3286385146419654, 'balanced_accuracy': 0.23569763865498294, 'top_3_accuracy': 0.7408278457196613, 'top_5_accuracy': 0.8664158043273753}
